# NVFP4 Verification Diagnostic

## Why did NVFP4-fixed give PPL 532 on OPT-13B?

The table note says: **"NVFP4-fixed collapses on padded opt-13b (fixed-clip artifact)"**.

This notebook isolates and quantifies the two failure modes:

### Bug 1 — Codebook not normalized to [0, 1]

`gf4_regularity_colab.py` passes the raw E2M1 magnitudes `{0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0}` to the GF4 activation quantizer, which clamps normalised magnitudes to **[0, 1]** before the codebook lookup. Five of the eight threshold values (1.25, 1.75, 2.5, 3.5, 5.0) exceed 1.0 and are **unreachable**, collapsing the effective codebook from 8 levels down to **3**.

```
mag = (x / scale).abs().clamp(0.0, 1.0)   ← clamp is here
idx = bucketize(mag, _THR)                ← thresholds {0.25, 0.75, 1.25, 1.75, 2.5, 3.5, 5.0}
```

Result: indices 0, 1, 2 only → reconstruction levels {0.0, 0.5, 1.0} × scale. 2-bit quality in a 4-bit container.

### Bug 2 — Fixed-clip / absmax scale (the "fixed-clip artifact")

The NVFP4 weight quantiser uses `scale = absmax / 6.0`. If the same absmax-style scale is applied to **activations**, extreme outliers (OPT-13B has channels with magnitude 100–1000× the RMS) dominate the scale, compressing all normal values into the lowest codebook levels.

The proper GF4 activation scheme searches over clip ratios {1.5, 2.0, 2.5, 3.0, 4.0} applied to the per-block RMS, which is robust to outliers after Hadamard rotation.

---

**Five schemes are compared end-to-end:**

| Scheme | Scale | Codebook | Expected |
|---|---|---|---|
| `gf4` | RMS × searched clip | Gaussian-quantile | Working baseline |
| `nvfp4_raw` | RMS × 2.5 | E2M1 raw (not normalised) | **Bug 1** — 3 effective levels → reproduces ~532 PPL |
| `nvfp4_norm` | RMS × 2.5 | E2M1 / 6 (normalised) | Bug 1 fix — all 8 levels |
| `nvfp4_absmax` | absmax / 6 | E2M1 / 6 (normalised) | **Bug 2** — fixed-clip artifact |
| `e2m1_rms_best` | RMS × 2.5 | E2M1 bias-searched | Best-effort NVFP4 for activations |

> **Runtime:** A100 40 GB recommended for OPT-13B. OPT-1.3B runs fine on T4.

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install -q transformers datasets accelerate scipy huggingface_hub

# ── 2. Clone repo (optional — only needed if you want the full CUDA_FP4_Test
#       code alongside this notebook; nvfp4_verify itself is self-contained.) ──
# REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'   # <-- EDIT
# SUBDIR   = 'Thesis_Compression/Python_Jenks_Test/Jenks_Tests/CUDA_FP4_Test'
# import os, subprocess
# _name = REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')
# if not os.path.isdir(_name):
#     subprocess.run(['git', 'clone', '--depth=1', REPO_URL], check=True)
# os.chdir(os.path.join(_name, SUBDIR))
# print('cwd:', os.getcwd())

In [ ]:
# ── 3. Configuration ──────────────────────────────────────────────────────────

# Model to evaluate. Use opt-1.3b for a quick smoke-test (~4 min on T4).
# Use opt-13b to reproduce the 532 PPL result from the table.
MODEL         = 'facebook/opt-1.3b'   # <-- change to facebook/opt-13b for the paper result

DEVICE_MAP    = False    # set True for opt-13b on a 40 GB GPU (uses accelerate)
HF_TOKEN      = ''       # set to 'hf_...' for gated models

EVAL_WINDOWS  = 10       # WikiText-2 test windows (use 50+ for paper-quality PPL)
CALIB_WINDOWS = 2        # windows for capturing calibration activations
SEQLEN        = 2048
HAD_BLOCK     = 32       # Hadamard block size
WBLOCK        = 16       # weight quantization block size
SEED          = 0
SKIP_PPL      = False    # set True to only run the level-histogram diagnostic
OUT_CSV       = 'nvfp4_verify_results.csv'

In [ ]:
# ── 4. Imports and codebook definitions ──────────────────────────────────────
import math, os, time, zlib
import numpy as np
import torch
import torch.nn as nn
from scipy.linalg import hadamard as _scipy_hadamard

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)

# ── Codebooks ────────────────────────────────────────────────────────────────
GF4_LEVEL = np.array(
    [0.0, 0.0796082, 0.1737177, 0.2828685,
     0.3952704, 0.5250730, 0.6961928, 1.0], dtype=np.float32)

# NVFP4 / E2M1 magnitudes as used in gf4_regularity_colab.py — raw, NOT normalised.
E2M1_MAG_RAW  = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0], dtype=np.float32)
# Normalised to [0, 1] (the correct form for the GF4-style activation quantiser).
E2M1_MAG_NORM = E2M1_MAG_RAW / 6.0   # {0, 0.0833, 0.1667, 0.25, 0.333, 0.5, 0.667, 1.0}

CLIP_RATIO = 2.5

# ── Hadamard ─────────────────────────────────────────────────────────────────
_H_CACHE = {}
def _hmat(block, device):
    k = (block, str(device))
    if k not in _H_CACHE:
        H = torch.tensor(_scipy_hadamard(block).astype(np.float32) / math.sqrt(block),
                         device=device)
        _H_CACHE[k] = H
    return _H_CACHE[k]

def rotate(x, signs, block=HAD_BLOCK):
    F_ = x.shape[-1]
    H  = _hmat(block, x.device)
    xr = (x * signs).reshape(*x.shape[:-1], F_ // block, block) @ H
    return xr.reshape(*x.shape[:-1], F_)

print('Imports OK.')

In [ ]:
# ── 5. Quantization schemes ───────────────────────────────────────────────────

def _quant_with_codebook(x, levels_np, scale_mode='rms'):
    """
    Generic activation quantiser used by all five schemes.

    scale_mode='rms'    : scale = block_rms * CLIP_RATIO  (our GF4 scheme)
    scale_mode='absmax' : scale = block_absmax / levels_np.max()  (fixed-clip / NVFP4 weight style)

    levels_np MUST be normalised to [0, 1] for scale_mode='rms'; values > 1
    produce thresholds that are unreachable after the clamp — that is Bug 1.
    """
    shp = x.shape
    xb  = x.reshape(-1, HAD_BLOCK).float()

    if scale_mode == 'rms':
        rms   = xb.pow(2).mean(-1, keepdim=True).add(1e-12).sqrt()
        scale = rms * CLIP_RATIO
    else:
        scale = xb.abs().amax(dim=-1, keepdim=True).clamp_min(1e-8) / float(levels_np.max())

    xn  = xb / scale
    # ↓ THIS CLAMP is the crux of Bug 1: values > 1 are unreachable.
    mag = xn.abs().clamp(0.0, 1.0)

    lv_t  = torch.tensor(levels_np, device=x.device)
    thr_t = (lv_t[:-1] + lv_t[1:]) / 2.0   # midpoint thresholds
    idx   = torch.bucketize(mag, thr_t)

    deq = lv_t[idx] * torch.sign(xn) * scale
    return deq.reshape(shp).to(x.dtype)


def quant_gf4(x):
    return _quant_with_codebook(x, GF4_LEVEL, scale_mode='rms')

def quant_nvfp4_raw(x):
    """Bug 1: raw E2M1_MAG into the [0,1]-clamped path -> only 3 effective levels."""
    return _quant_with_codebook(x, E2M1_MAG_RAW, scale_mode='rms')

def quant_nvfp4_norm(x):
    """Bug 1 fix: normalise to [0,1] first -> all 8 levels reachable."""
    return _quant_with_codebook(x, E2M1_MAG_NORM, scale_mode='rms')

def quant_nvfp4_absmax(x):
    """Bug 2 (fixed-clip artifact): absmax/6 scale instead of RMS scale."""
    return _quant_with_codebook(x, E2M1_MAG_NORM, scale_mode='absmax')

def quant_e2m1_rms_best(x):
    """Best-effort E2M1 on activations: RMS scale, bias {0,1,2} searched per block."""
    CB = {
        0: np.array([1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0], dtype=np.float32),
        1: np.array([0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0,  6.0], dtype=np.float32),
        2: np.array([0.25, 0.375, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0], dtype=np.float32),
    }
    shp = x.shape
    xb  = x.reshape(-1, HAD_BLOCK).float()
    rms = xb.pow(2).mean(-1, keepdim=True).add(1e-12).sqrt()
    scale = rms * CLIP_RATIO
    best_mse = torch.full((xb.shape[0],), float('inf'), device=x.device)
    best_deq = torch.zeros_like(xb)
    for b, cb_np in CB.items():
        cb_max = float(cb_np.max())
        xn     = (xb / scale) * cb_max
        lv_t   = torch.tensor(cb_np, device=x.device)
        thr_t  = (lv_t[:-1] + lv_t[1:]) / 2.0
        idx    = torch.bucketize(xn.abs().clamp(0.0, cb_max), thr_t)
        deq_b  = lv_t[idx] * torch.sign(xn) * (scale / cb_max)
        mse_b  = ((xb - deq_b) ** 2).mean(dim=-1)
        better = mse_b < best_mse
        best_mse = torch.where(better, mse_b, best_mse)
        best_deq = torch.where(better.unsqueeze(-1), deq_b, best_deq)
    return best_deq.reshape(shp).to(x.dtype)


SCHEMES = {
    'gf4':           quant_gf4,
    'nvfp4_raw':     quant_nvfp4_raw,    # Bug 1: 3 effective levels
    'nvfp4_norm':    quant_nvfp4_norm,   # Bug 1 fix
    'nvfp4_absmax':  quant_nvfp4_absmax, # Bug 2: fixed-clip artifact
    'e2m1_rms_best': quant_e2m1_rms_best,
}
print('Quantization schemes defined:', list(SCHEMES.keys()))

In [ ]:
# ── 6. Load model ─────────────────────────────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

print(f'Loading {MODEL}...')
tok     = AutoTokenizer.from_pretrained(MODEL)
load_kw = dict(torch_dtype=torch.float16)
if DEVICE_MAP:
    load_kw['device_map'] = 'auto'

model = AutoModelForCausalLM.from_pretrained(MODEL, **load_kw)
if not DEVICE_MAP:
    model = model.cuda()
model.eval()

_input_dev = (model.get_input_embeddings().weight.device
              if hasattr(model, 'get_input_embeddings')
              else next(model.parameters()).device)
print(f'Input embedding device: {_input_dev}')
print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# ── 7. Load WikiText-2 windows ────────────────────────────────────────────────
def _load_wikitext(split):
    for repo in ('Salesforce/wikitext', 'wikitext'):
        try:
            return load_dataset(repo, 'wikitext-2-raw-v1', split=split)
        except Exception:
            pass
    raise RuntimeError('Could not load wikitext-2-raw-v1')

def get_windows(ids, seqlen, n):
    n = min(n, ids.numel() // seqlen)
    return [ids[i * seqlen:(i + 1) * seqlen].unsqueeze(0) for i in range(n)]

test_ids  = tok('\n\n'.join(_load_wikitext('test')['text']),
                return_tensors='pt').input_ids[0]
train_ids = tok('\n\n'.join(_load_wikitext('train')['text'][:2000]),
                return_tensors='pt').input_ids[0]

eval_wins  = get_windows(test_ids,  SEQLEN, EVAL_WINDOWS)
calib_wins = get_windows(train_ids, SEQLEN, CALIB_WINDOWS)
print(f'Eval windows: {len(eval_wins)}  |  Calib windows: {len(calib_wins)}')

In [ ]:
# ── 8. FP16 baseline perplexity ───────────────────────────────────────────────
@torch.no_grad()
def perplexity(mdl):
    nll = ntok = 0
    in_dev = (mdl.get_input_embeddings().weight.device
              if hasattr(mdl, 'get_input_embeddings')
              else next(mdl.parameters()).device)
    for w in eval_wins:
        w   = w.to(in_dev)
        out = mdl(w, labels=w)
        nll  += out.loss.float().item() * (w.numel() - 1)
        ntok += w.numel() - 1
    return math.exp(nll / ntok)

ppl_fp16 = perplexity(model)
print(f'FP16 baseline PPL: {ppl_fp16:.4f}')

In [ ]:
# ── 9. Identify target layers + capture calibration activations ───────────────
RETAIN = ('fc2', 'down_proj', 'lm_head')

targets = [
    (n, m) for n, m in model.named_modules()
    if isinstance(m, nn.Linear)
    and not any(s in n for s in RETAIN)
    and m.in_features % HAD_BLOCK == 0
]
print(f'Target layers: {len(targets)}')

captured_raw = {n: [] for n, _ in targets}
_hooks       = []

def _make_capture(name):
    def _h(mod, inp, _out):
        captured_raw[name].append(
            inp[0].detach().reshape(-1, inp[0].shape[-1]).float().cpu())
    return _h

for name, mod in targets:
    _hooks.append(mod.register_forward_hook(_make_capture(name)))

with torch.no_grad():
    for w in calib_wins:
        model(w.to(_input_dev))

for h in _hooks:
    h.remove()
print('Calibration activations captured.')

In [ ]:
# ── 10. Per-layer level-histogram diagnostic ──────────────────────────────────
# This is the smoking gun: nvfp4_raw will show n_levels_used = 3, not 8.

from collections import defaultdict

print(f"\n{'Layer':<42} {'Scheme':<16} {'n_lv':>5} {'rel_mse':>9} {'clip':>7}  "
      f"Level histogram [0..7]")
print('-' * 115)

layer_rows = []
for i, (name, _) in enumerate(targets):
    if not captured_raw.get(name):
        continue
    seed  = zlib.crc32(name.encode('utf-8')) % (2 ** 31)
    gen   = torch.Generator().manual_seed(seed)
    K     = targets[i][1].in_features
    signs = torch.randint(0, 2, (K,), generator=gen).float() * 2 - 1

    X_raw = torch.cat(captured_raw[name], dim=0)
    X_had = rotate(X_raw, signs, HAD_BLOCK)

    flat = X_had.reshape(-1).numpy()
    if flat.shape[0] > 262144:
        rng  = np.random.default_rng(0)
        flat = rng.choice(flat, 262144, replace=False)

    x   = torch.tensor(flat, dtype=torch.float32)
    var_x = float(x.pow(2).mean())
    short = name if len(name) <= 42 else name[:42]

    for scheme_name, fn in SCHEMES.items():
        x_hat   = fn(x)
        rel_mse = float(((x - x_hat) ** 2).mean()) / (var_x + 1e-12)

        # Recover codebook index for each element to build histogram.
        xb  = x.reshape(-1, HAD_BLOCK)
        rms = xb.pow(2).mean(-1, keepdim=True).add(1e-12).sqrt()
        if scheme_name == 'nvfp4_absmax':
            scale = xb.abs().amax(dim=-1, keepdim=True).clamp_min(1e-8) / 6.0
            lv_np = E2M1_MAG_NORM
        elif scheme_name == 'nvfp4_raw':
            scale = rms * CLIP_RATIO
            lv_np = E2M1_MAG_RAW   # <-- raw levels → thresholds > 1 unreachable
        elif scheme_name in ('nvfp4_norm', 'e2m1_rms_best'):
            scale = rms * CLIP_RATIO
            lv_np = E2M1_MAG_NORM
        else:  # gf4
            scale = rms * CLIP_RATIO
            lv_np = GF4_LEVEL

        xn    = (xb / scale).abs().clamp(0.0, 1.0)
        lv_t  = torch.tensor(lv_np)
        thr_t = (lv_t[:-1] + lv_t[1:]) / 2.0
        idx   = torch.bucketize(xn, thr_t).reshape(-1).numpy()
        hist  = np.bincount(idx, minlength=8)
        n_lv  = int((hist > 0).sum())
        clip  = float(hist[-1]) / max(len(idx), 1)

        marker = '  <-- BUG 1: only 3 levels!' if (scheme_name == 'nvfp4_raw' and n_lv <= 3) else ''
        hist_s = ' '.join(f'{c:6d}' for c in hist)
        print(f'{short:<42} {scheme_name:<16} {n_lv:>5}  '
              f'{rel_mse:>8.4f}  {clip:>6.3f}  [{hist_s}]{marker}')
        layer_rows.append(dict(layer=name, scheme=scheme_name,
                               n_levels=n_lv, rel_mse=rel_mse, clip_frac=clip))
    print()

    # Free as we go to avoid OOM on opt-13b.
    del captured_raw[name], X_raw, X_had

In [ ]:
# ── 11. Aggregate diagnostic summary ─────────────────────────────────────────
agg = defaultdict(list)
for row in layer_rows:
    agg[row['scheme']].append(row)

EXPECTED = {
    'gf4':           'all 8 levels, low MSE                 [the working baseline]',
    'nvfp4_raw':     'only 3 levels, HIGH MSE               [Bug 1 — reproduces 532 PPL]',
    'nvfp4_norm':    'all 8 levels, moderate MSE            [Bug 1 fix]',
    'nvfp4_absmax':  'all 8 levels, higher MSE              [Bug 2 — fixed-clip artifact]',
    'e2m1_rms_best': 'all 8 levels, best NVFP4 MSE          [ideal NVFP4 on activations]',
}

print(f"\n{'Scheme':<18} {'mean n_lv':>10} {'mean rel_mse':>14} {'mean clip':>10}  Note")
print('-' * 90)
for scheme in SCHEMES:
    rows = agg[scheme]
    if not rows:
        continue
    mn_lv  = np.mean([r['n_levels'] for r in rows])
    mn_mse = np.mean([r['rel_mse']  for r in rows])
    mn_cl  = np.mean([r['clip_frac'] for r in rows])
    print(f'{scheme:<18} {mn_lv:>10.1f} {mn_mse:>14.6f} {mn_cl:>10.4f}  '
          f'{EXPECTED.get(scheme, "")}')

In [ ]:
# ── 12. End-to-end perplexity per scheme ─────────────────────────────────────
# (skip with SKIP_PPL = True to just look at the level histograms)

import csv

ppl_table = [('fp16_baseline', ppl_fp16, 0.0)]

if SKIP_PPL:
    print('[SKIP_PPL=True] Skipping end-to-end PPL evaluation.')
else:
    def _patch_model(act_fn):
        """Install NVFP4 weight quant + activation hook; return restore callback."""
        def quant_e4m3(s):
            s = s.clamp(min=2.0 ** -9, max=448.0)
            e = torch.floor(torch.log2(s))
            m = torch.round((s / torch.exp2(e) - 1.0) * 8.0) / 8.0
            return (1.0 + m) * torch.exp2(e)

        orig_weights = {}
        for name, mod in targets:
            dev  = mod.weight.device
            seed = zlib.crc32(name.encode('utf-8')) % (2 ** 31)
            gen  = torch.Generator().manual_seed(seed)
            K    = mod.in_features
            signs = (torch.randint(0, 2, (K,), generator=gen).float() * 2 - 1).to(dev)

            orig_weights[name] = mod.weight.data.clone()
            with torch.no_grad():
                # Rotate + quantize in 1024-row chunks: avoids materialising
                # a full float32 W_rot + float32 intermediates simultaneously.
                # Peak per chunk ~30 MB regardless of layer size.
                W_fp16 = mod.weight.data
                M_rows, K_cols = W_fp16.shape
                blk = WBLOCK if K_cols % WBLOCK == 0 else (32 if K_cols % 32 == 0 else HAD_BLOCK)
                _mag = torch.tensor(E2M1_MAG_RAW, device=dev)
                _thr = (_mag[:-1] + _mag[1:]) / 2.0
                W_q = torch.empty_like(W_fp16)
                for _i in range(0, M_rows, 1024):
                    _ch   = W_fp16[_i:_i+1024].float()
                    _ch_r = rotate(_ch, signs, HAD_BLOCK); del _ch
                    _Wb   = _ch_r.reshape(-1, blk)
                    _amax = _Wb.abs().amax(1, keepdim=True).clamp_min(1e-8)
                    _sc   = quant_e4m3(_amax / 6.0)
                    _wn   = _Wb / _sc
                    _idx  = torch.bucketize(_wn.abs().clamp(0.0, 6.0), _thr)
                    W_q[_i:_i+1024] = (_mag[_idx] * torch.sign(_wn) * _sc).reshape(-1, K_cols).to(W_fp16.dtype)
                    del _ch_r, _Wb, _amax, _sc, _wn, _idx
                mod.weight.data.copy_(W_q)
                del W_q; torch.cuda.empty_cache()

            def _pre(mod_, inp_, _s=signs, _fn=act_fn):
                x  = inp_[0]
                xr = rotate(x.float().to(_s.device), _s, HAD_BLOCK)
                xq = _fn(xr).to(x.dtype)
                return (xq.to(x.device),) + inp_[1:]

            if not hasattr(mod, '_verify_hooks'):
                mod._verify_hooks = []
            mod._verify_hooks.append(mod.register_forward_pre_hook(_pre))

        def _restore():
            for name, mod in targets:
                mod.weight.data.copy_(orig_weights[name].to(mod.weight.device))
                for h in getattr(mod, '_verify_hooks', []):
                    h.remove()
                mod._verify_hooks = []
        return _restore

    for scheme_name, act_fn in SCHEMES.items():
        print(f'  [{scheme_name}] measuring PPL...', end=' ', flush=True)
        restore = _patch_model(act_fn)
        t0  = time.time()
        ppl = perplexity(model)
        restore()
        delta = ppl - ppl_fp16
        print(f'PPL = {ppl:.4f}  ({delta:+.4f} vs FP16)  [{time.time()-t0:.1f}s]')
        ppl_table.append((scheme_name, ppl, delta))


In [ ]:
# ── 13. Summary table + CSV ───────────────────────────────────────────────────
NOTES = {
    'fp16_baseline': '',
    'gf4':           'working baseline',
    'nvfp4_raw':     '<-- Bug 1: 3 effective levels (fixed-clip + no normalisation)',
    'nvfp4_norm':    'Bug 1 fix: normalise E2M1_MAG to [0,1]',
    'nvfp4_absmax':  'Bug 2: fixed-clip artifact (absmax dominates outlier channels)',
    'e2m1_rms_best': 'best-effort NVFP4 for activations',
}

print(f'\n{"="*80}')
print(f'PERPLEXITY SUMMARY  --  {MODEL}  --  {len(eval_wins)} windows x {SEQLEN} tok')
print(f'{"="*80}')
print(f"\n{'Scheme':<20} {'PPL':>10} {'dPPL(fp16)':>12}  Note")
print('-' * 75)
for name, ppl, delta in ppl_table:
    label = 'fp16 baseline' if name == 'fp16_baseline' else name
    print(f'{label:<20} {ppl:>10.4f} {delta:>+12.4f}  {NOTES.get(name, "")}')

print("""
┌─ INTERPRETATION ─────────────────────────────────────────────────────────┐
│ nvfp4_raw PPL >> nvfp4_norm PPL  →  Bug 1 confirmed.                    │
│   Raw E2M1_MAG thresholds > 1.0 are unreachable after the [0,1] clamp;  │
│   only 3 of 8 codebook levels are ever assigned.                         │
│                                                                           │
│ nvfp4_absmax PPL >> nvfp4_norm PPL  →  Bug 2 confirmed.                 │
│   absmax/6 scale is dominated by OPT-13B outlier channels; most values  │
│   compress to level 0.  This is the "fixed-clip artifact" in the table. │
│                                                                           │
│ THE FIX (one line): set_codebook(E2M1_MAG / 6.0, device)                │
│   and ensure activation scale uses RMS + searched clip, not absmax.      │
└───────────────────────────────────────────────────────────────────────────┘
""")

import csv
new_file = not os.path.exists(OUT_CSV)
with open(OUT_CSV, 'a', newline='') as f:
    w = csv.writer(f)
    if new_file:
        w.writerow(['timestamp', 'model', 'scheme', 'ppl', 'dppl_fp16',
                    'eval_windows', 'calib_windows', 'seqlen'])
    ts = time.strftime('%Y-%m-%d %H:%M:%S')
    for name, ppl, delta in ppl_table:
        w.writerow([ts, MODEL, name, f'{ppl:.4f}', f'{delta:.4f}',
                    len(eval_wins), len(calib_wins), SEQLEN])
print(f'Results written to {OUT_CSV}')